# NTP and SNTP — keeping device clocks honest

A GPS-referenced NTP server and a device with a drifting clock share an in-memory network with 10 ms each way. The same `SntpClient` measures your computer against `pool.ntp.org` with `UseServer("pool.ntp.org")`; it never changes the system clock.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.20.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.20.0-preview.1"

## Timestamps and packets
NTP counts seconds from 1900 in 32 bits plus a 32-bit fraction; the seconds wrap in 2036 and the era rule maps them back.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Ntp;
using IoTCom.Net.Transports;
using System.Net;

var unix = NtpTimestamp.FromDateTime(new DateTime(1970, 1, 1, 0, 0, 0, DateTimeKind.Utc));
Console.WriteLine($"1970-01-01 = 0x{unix.Raw:X16}");
Console.WriteLine($"0x0000000100000000 = {new NtpTimestamp(1UL << 32)}  (era 1)");
var request = new NtpPacket { Mode = NtpMode.Client, Transmit = NtpTimestamp.FromDateTime(DateTime.UtcNow) }.Encode();
foreach (var f in NtpPacket.Describe(request)) Console.WriteLine($"{f.Name,-11} {f.Value}");

## One exchange: offset and delay
T1 leaves the device, T2 reaches the server, T3 leaves the server, T4 arrives back.

In [ ]:
var ntpNet = new InMemoryDatagramNetwork { Latency = TimeSpan.FromMilliseconds(10) };
var ntpAddress = new IPEndPoint(IPAddress.Parse("10.0.0.1"), 123);
var gpsServer = NtpServer.Create(o => o.UseInMemory(ntpNet, ntpAddress).WithReference("GPS"));
await gpsServer.StartAsync();
var deviceClock = new DriftingClock(TimeSpan.FromSeconds(-3.2), driftPpm: 20_000);   // 2 %: exaggerated
var sntp = SntpClient.Create(o => { o.UseInMemory(ntpNet); o.UseServer(ntpAddress); o.Clock = () => deviceClock.UtcNow; o.MinimumPollInterval = TimeSpan.Zero; });
var first = await sntp.QueryAsync(ntpAddress);
Console.WriteLine($"T1 {first.T1}\nT2 {first.T2}\nT3 {first.T3}\nT4 {first.T4}");
Console.WriteLine($"offset {first.Offset.TotalMilliseconds:+0.0} ms, delay {first.RoundTripDelay.TotalMilliseconds:0.0} ms");

## Discipline the clock
Step the device clock by the measured offset and watch the error stay small between synchronisations.

In [ ]:
deviceClock.Step(first.Offset);
for (var i = 0; i < 3; i++)
{
    await Task.Delay(500);
    var before = deviceClock.Error;
    var r = await sntp.QueryAsync(ntpAddress);
    deviceClock.Step(r.Offset);
    Console.WriteLine($"drifted {before.TotalMilliseconds:+0.0} ms, corrected to {deviceClock.Error.TotalMilliseconds:+0.0} ms");
}

## When the server cannot be trusted
A server that lost its reference advertises leap 3; a busy one answers kiss-o'-death RATE. Both are refused.

In [ ]:
gpsServer.Leap = NtpLeap.Unsynchronised;
try { await sntp.QueryAsync(ntpAddress); } catch (DeviceException e) { Console.WriteLine(e.Message); }
var busyAddress = new IPEndPoint(IPAddress.Parse("10.0.0.2"), 123);
var busy = NtpServer.Create(o => { o.UseInMemory(ntpNet, busyAddress); o.RateLimit = TimeSpan.FromSeconds(10); });
await busy.StartAsync();
await sntp.QueryAsync(busyAddress);
try { await sntp.QueryAsync(busyAddress); } catch (NtpKissOfDeathException e) { Console.WriteLine($"{e.Code}: {e.Message}"); }
await sntp.DisposeAsync();
await busy.DisposeAsync();
await gpsServer.DisposeAsync();

## Going further
`iotcom ntp query`, `iotcom ntp query --sim --frames`, `iotcom ntp serve`, the Gallery's *Fleet clock sync over NTP* and the NtpClock sample. See `docs/en/protocols/ntp.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*